# 🌦️ Pipeline de Análisis de Datos Meteorológicos con DuckDB e SQL
**Curso:** Data Science II - Coderhouse  
**Alumno/a:** Paulo Martino  
**Entorno:** Google Colab / Jupyter Notebook

---  
## 📌 Descripción
Este Notebook ejecuta un pipeline analítico completo utilizando **DuckDB** como motor OLAP embebido en memoria. Procesamos un dataset meteorológico (2012 - 2015) directamente sobre archivos CSV y Parquet sin requerir servidores de base de datos tradicionales.

### 1. Instalación de dependencias e importación de librerías

In [ ]:
!pip install duckdb pandas matplotlib -q
import os
import time
import urllib.request
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

### 2. Descarga y preparación de los datos en Google Colab

In [ ]:
os.makedirs('data', exist_ok=True)
csv_path = 'data/weather_data.csv'
parquet_path = 'data/weather_data.parquet'

if not os.path.exists(csv_path):
    url = 'https://raw.githubusercontent.com/vega/vega-datasets/main/data/seattle-weather.csv'
    print(f'📥 Descargando {csv_path} desde fuente pública...')
    urllib.request.urlretrieve(url, csv_path)
    print('✅ Descarga completada.')
else:
    print('✅ Archivo CSV ya disponible localmente.')

### 3. Conexión a DuckDB y Creación de Vista Virtual (CSV & Parquet)

In [ ]:
con = duckdb.connect(database=':memory:')

# Creación de vista con tratamiento de nulos vía COALESCE
query_vista = '''
CREATE VIEW tabla_clima AS 
SELECT 
    date AS fecha,
    COALESCE(precipitation, 0) AS precipitacion,
    COALESCE(temp_max, 0) AS temp_maxima,
    COALESCE(temp_min, 0) AS temp_minima,
    COALESCE(wind, 0) AS viento,
    COALESCE(weather, 'Sin datos') AS clima
FROM read_csv_auto('data/weather_data.csv');
'''
con.execute(query_vista)
print('✅ Vista virtual "tabla_clima" registrada correctamente.')

# Exportación a Parquet
con.execute(f"COPY (SELECT * FROM tabla_clima) TO '{parquet_path}' (FORMAT PARQUET);")
print('✅ Archivo Parquet generado en "data/weather_data.parquet".')

### 4. Consulta 1: Promedios Mensuales de Temperatura y Precipitaciones

In [ ]:
query1 = '''
SELECT 
    STRFTIME(CAST(fecha AS DATE), '%Y-%m') AS mes_anio,
    COUNT(*) AS dias_registrados,
    ROUND(AVG(temp_maxima), 2) AS promedio_temp_max,
    ROUND(AVG(temp_minima), 2) AS promedio_temp_min,
    ROUND(SUM(precipitacion), 2) AS total_precipitacion
FROM tabla_clima
GROUP BY mes_anio
ORDER BY mes_anio;
'''
df_mensual = con.execute(query1).df()
df_mensual.head(12)

### 5. Consulta 2: Extremos Meteorológicos por Condición de Clima

In [ ]:
query2 = '''
SELECT 
    clima,
    COUNT(*) AS cantidad_dias,
    MAX(temp_maxima) AS temp_maxima_historica,
    MIN(temp_minima) AS temp_minima_historica,
    MAX(viento) AS viento_maximo
FROM tabla_clima
GROUP BY clima
ORDER BY cantidad_dias DESC;
'''
df_clima = con.execute(query2).df()
df_clima

### 6. Consulta 3: Días de Condiciones Extremas

In [ ]:
query3 = '''
SELECT 
    fecha,
    clima,
    temp_maxima,
    temp_minima,
    precipitacion,
    viento
FROM tabla_clima
WHERE temp_maxima >= 25.0 OR precipitacion >= 20.0
ORDER BY temp_maxima DESC;
'''
df_extremos = con.execute(query3).df()
df_extremos.head(10)

### 7. Gráfico de Evolución Temporal de Temperatura y Lluvias

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

color = 'tab:red'
ax1.set_xlabel('Mes (Año-Mes)', fontsize=12)
ax1.set_ylabel('Temp Máx Promedio (°C)', color=color, fontsize=12)
ax1.plot(df_mensual['mes_anio'], df_mensual['promedio_temp_max'], color=color, marker='o', linewidth=2, label='Temp Máxima')
ax1.tick_params(axis='y', labelcolor=color)
plt.xticks(rotation=60)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Precipitación Total (mm)', color=color, fontsize=12)
ax2.bar(df_mensual['mes_anio'], df_mensual['total_precipitacion'], color=color, alpha=0.3, label='Precipitación')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Evolución Mensual de Temperaturas y Lluvias Acumuladas (DuckDB Analytics)', fontsize=14, pad=15)
fig.tight_layout()
plt.show()

### 8. Benchmark de Velocidad (CSV vs Parquet en DuckDB)

In [ ]:
t0 = time.time()
con.execute("SELECT COUNT(*) FROM read_csv_auto('data/weather_data.csv')").fetchall()
t_csv = (time.time() - t0) * 1000

t0 = time.time()
con.execute("SELECT COUNT(*) FROM read_parquet('data/weather_data.parquet')").fetchall()
t_parquet = (time.time() - t0) * 1000

print(f'⏱️ Consulta sobre CSV:     {t_csv:.2f} ms')
print(f'⏱️ Consulta sobre Parquet: {t_parquet:.2f} ms')
if t_parquet > 0:
    print(f'🚀 Parquet es {(t_csv / t_parquet):.2f}x más rápido que CSV.')